# process_raw.ipynb

This notebook reads in the raw ACS PUMA data and performs cleaning on it to serve as final refdata for PUMA statistics. It also performs aggregation on the PUMA data to get statistics for MIGPUMAs.

In [24]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))
from lib import io as lio

In [25]:
one_year_acs = "R50171584_SL795.csv"
five_year_acs = "R50171626_SL795.csv"
table_csv = "acs_tables_2018.txt"
year = 2018

In [26]:
acs1_code_to_name = pd.read_fwf(
    "./raw/" + table_csv, colspecs=[(0, 8), (8, None)]
).set_index("Table")
acs1_code_to_name

,Title
Table,
A00001,Total Population
A00002,Population Density (Per Sq. Mile)
A00003,Land Area (Sq. Miles)
A02001,Sex
A02002,Sex by Age
...,...
A20002,Health Insurance by Age
A10048,Occupied Housing Units by Complete Facilities
A10033B,Complete Facilities (Renter-Occupied Housing U...


In [27]:
acs1_df = pd.read_csv("./raw/" + one_year_acs, header=[0, 1], index_col=0)
acs1_df.columns = [
    ".".join(str(level.replace(":", "")) for level in col) for col in acs1_df.columns
]
acs1_df.index = acs1_df.index.astype(str).str.zfill(7)
acs1_df.index.name = "PUMA"
acs1_df = acs1_df[
    ~acs1_df["State/U.S.-Abbreviation (USPS).Geo_STUSAB"].isin(["ak", "hi", "pr"])
]
acs1_df.head()

,Geographic Identifier.Geo_GEOID,Name of Area.Geo_NAME,Qualifying Name.Geo_QName,State/U.S.-Abbreviation (USPS).Geo_STUSAB,Summary Level.Geo_SUMLEV,Geographic GEOCOMP.Geo_GEOCOMP,File Identification.Geo_FILEID,Logical Record Number.Geo_LOGRECNO,US.Geo_US,Region.Geo_REGION,...,Occupied Housing Units with Telephone Service Available Householder 15 to 34 Years.SE_A10029_003,Occupied Housing Units with Telephone Service Available Householder 35 to 64 Years.SE_A10029_004,Occupied Housing Units with Telephone Service Available Householder 65 Years and Over.SE_A10029_005,Occupied Housing Units No Telephone Service Available.SE_A10029_006,Occupied Housing Units No Telephone Service Available Householder 15 to 34 Years.SE_A10029_007,Occupied Housing Units No Telephone Service Available Householder 35 to 64 Years.SE_A10029_008,Occupied Housing Units No Telephone Service Available Householder 65 Years and Over.SE_A10029_009,Renter-Occupied Housing Units.SE_A10061B_001,Renter-Occupied Housing Units with Telephone Service Available.SE_A10061B_002,Renter-Occupied Housing Units No Telephone Service Available.SE_A10061B_003
PUMA,,,,,,,,,,,,,,,,,,,,,
0100100,79500US0100100,"Lauderdale, Colbert, Franklin & Marion (Northe...","Lauderdale, Alabama",al,795,0,ACSSF,57,NaN,NaN,...,12763.0,34075.0,23005.0,3888.0,987.0,2198.0,703.0,23052.0,20665.0,2387.0
0100200,79500US0100200,Limestone & Madison (Outer) Counties--Huntsvil...,Limestone & Madison (Outer) Counties--Huntsvil...,al,795,0,ACSSF,58,NaN,NaN,...,12596.0,42233.0,16872.0,759.0,0.0,601.0,158.0,16978.0,16355.0,623.0
0100301,79500US0100301,Huntsville (North) & Madison (East) Cities PUM...,Huntsville (North) & Madison (East) Cities PUM...,al,795,0,ACSSF,59,NaN,NaN,...,13001.0,30361.0,12674.0,701.0,148.0,191.0,362.0,22396.0,22036.0,360.0
0100302,79500US0100302,"Huntsville City (Central & South) PUMA, Alabama","Huntsville City (Central & South) PUMA, Alabama",al,795,0,ACSSF,60,NaN,NaN,...,9463.0,22143.0,11687.0,583.0,121.0,350.0,112.0,18503.0,18079.0,424.0
0100400,79500US0100400,"DeKalb & Jackson Counties PUMA, Alabama","DeKalb & Jackson Counties PUMA, Alabama",al,795,0,ACSSF,61,NaN,NaN,...,7812.0,24853.0,13875.0,609.0,214.0,309.0,86.0,12206.0,12053.0,153.0


In [28]:
# lower 48 + dc
print(acs1_df["State/U.S.-Abbreviation (USPS).Geo_STUSAB"].nunique())

49


In [29]:
print(acs1_df.index.is_unique, acs1_df.columns.is_unique)

True True


In [30]:
acs5_df = pd.read_csv("./raw/" + five_year_acs, header=[0, 1], index_col=0)
acs5_df.columns = [
    ".".join(str(level.replace(":", "")) for level in col) for col in acs5_df.columns
]
acs5_df.index = acs5_df.index.astype(str).str.zfill(7)
acs5_df.index.name = "PUMA"
acs5_df = acs5_df[
    ~acs5_df["State/U.S.-Abbreviation (USPS).Geo_STUSAB"]
    .str.lower()
    .isin(["ak", "hi", "pr"])
]
acs5_df.head()

,Name of Area.Geo_NAME,Qualified Area Name.Geo_qname,State/U.S.-Abbreviation (USPS).Geo_STUSAB,Summary Level.Geo_SUMLEV,Geographic Component.Geo_GEOCOMP,File Identification.Geo_FILEID,Logical Record Number.Geo_LOGRECNO,US.Geo_US,Region.Geo_REGION,Division.Geo_DIVISION,...,Own Children under 18 Years.SE_A10065_001,Own Children under 18 Years Children Living with Single Parents.SE_A10065_002,Households.SE_A10066_001,Households 1-Person Household.SE_A10066_002,Households 2-Person Household.SE_A10066_003,Households 3-Person Household.SE_A10066_004,Households 4-Person Household.SE_A10066_005,Households 5-Person Household.SE_A10066_006,Households 6-Person Household.SE_A10066_007,Households 7-or-More Person Household.SE_A10066_008
PUMA,,,,,,,,,,,,,,,,,,,,,
0100100,Lauderdale,"Lauderdale, Colbert, Franklin & Marion (Northe...",AL,795,0,ACSSF,8665,NaN,NaN,NaN,...,34024,10838,74643,21789,28179,11226,8359,3640,920,530
0100200,Limestone & Madison (Outer) Counties--Huntsvil...,Limestone & Madison (Outer) Counties--Huntsvil...,AL,795,0,ACSSF,8666,NaN,NaN,NaN,...,39583,8678,69828,16998,25155,11558,10273,4187,1240,417
0100301,Huntsville (North) & Madison (East) Cities PUMA,Huntsville (North) & Madison (East) Cities PUM...,AL,795,0,ACSSF,8667,NaN,NaN,NaN,...,24780,7219,52260,17199,16648,8358,6166,2525,984,380
0100302,Huntsville City (Central & South) PUMA,"Huntsville City (Central & South) PUMA, Alabama",AL,795,0,ACSSF,8668,NaN,NaN,NaN,...,18991,7424,43997,16156,15219,5878,4280,1693,520,251
0100400,DeKalb & Jackson Counties PUMA,"DeKalb & Jackson Counties PUMA, Alabama",AL,795,0,ACSSF,8669,NaN,NaN,NaN,...,24070,7277,46758,12733,16828,7197,5944,2653,809,594


In [31]:
# lower 48 + dc
print(acs5_df["State/U.S.-Abbreviation (USPS).Geo_STUSAB"].nunique())

49


In [32]:
print(acs5_df.index.is_unique, acs5_df.columns.is_unique)

True True


In [33]:
# compare puma differences between 1-year and 5-year ACS data
# for cross-decade migration analysis, the PUMA definitions may change
print(set(acs5_df.index).symmetric_difference(set(acs1_df.index)))

set()


In [34]:
# use the 5-year ACS data to fill in missing data in the 1-year ACS data
# the 1-year ACS data has a lot more nas since for small populations, data is ommitted due to small sample sizes

# create a dictionary to rename the columns in the 5-year ACS data to match the 1-year ACS data using the underlying codes which should be constant
acs1_df_se_code_to_col = dict()
for col in acs1_df.columns:
    segments = col.split(".")
    se_code = segments[-1]
    desc = "".join(segments[:-1])
    acs1_df_se_code_to_col[se_code] = (col, desc)


rename_map = {}
no_match = []
for col in acs5_df.columns:
    segments = col.split(".")
    acs5_se_code = segments[-1]
    acs5_desc = "".join(segments[:-1])
    # do not rename geo columns, do not care about these
    if "Geo" in acs5_se_code:
        continue

    match_found = False
    for se_code, (acs1_col, asc1_desc) in acs1_df_se_code_to_col.items():
        if se_code == acs5_se_code:
            # NOTE: the descriptions can be off, but that is likely due to social explorer just naming things differently
            # a cursory search showed that the description differences don't mean anything
            print(f"{acs5_desc} -> {asc1_desc}")
            rename_map[col] = acs1_col
            match_found = True
            break  # stop at first match to avoid overwriting on multiple hits

    if not match_found:
        no_match.append(col)

print()
for col in no_match:
    print(col)

# only keep the columns in the 5-year ACS data that have a match in the 1-year ACS data
acs5_df_cleaned = acs5_df.rename(columns=rename_map)
acs1_df_cols = set(acs1_df.columns)
acs5_df_cleaned = acs5_df_cleaned[
    [col for col in acs5_df_cleaned.columns if col in acs1_df_cols]
]

Total Population -> Total Population
Total Population -> Total Population
Population Density (Per Sq Mile) -> Population Density (Per Sq Mile)
Area (Land) -> Area (Land)
Area Total -> Area Total
Area Total Area (Land) -> Area Total Area (Land)
Area Total Area (Water) -> Area Total Area (Water)
Total Population -> Total Population
Total Population Male -> Total Population Male
Total Population Female -> Total Population Female
Total Population -> Total Population
Total Population Male -> Total Population Male
Total Population Male Under 5 Years -> Total Population Male Under 5 Years
Total Population Male 5 to 9 Years -> Total Population Male 5 to 9 Years
Total Population Male 10 to 14 Years -> Total Population Male 10 to 14 Years
Total Population Male 15 to 17 Years -> Total Population Male 15 to 17 Years
Total Population Male 18 to 24 Years -> Total Population Male 18 to 24 Years
Total Population Male 25 to 34 Years -> Total Population Male 25 to 34 Years
Total Population Male 35 to 44

In [35]:
puma_df = acs1_df.combine_first(acs5_df)

In [36]:
# look at effect of combine first the 1-year ACS data with the 5-year ACS data
before_na = acs1_df.isna()
after_na = puma_df.isna()
filled_mask = before_na & ~after_na

filled_cols = filled_mask.any(axis=0)
for col in filled_cols[filled_cols].index.tolist():
    print(f"{col}: {before_na[col].sum()} -> {after_na[col].sum()}")
print()
print(acs1_df.isna().sum().sum(), puma_df.isna().sum().sum())

2.00 to 2.99.SE_C13004_003: 257 -> 0
3.00 to 3.99.SE_C13004_004: 257 -> 0
4.00 to 4.99.SE_C13004_005: 257 -> 0
5.00 and Over.SE_C13004_006: 257 -> 0
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) American Indian and Alaska Native Alone  Householder.SE_A14019_004: 1633 -> 109
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Asian Alone Householder.SE_A14019_005: 396 -> 3
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Black or African American Alone Householder.SE_A14019_003: 227 -> 5
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Native Hawaiian and Other Pacific Islander Alone  Householder.SE_A14019_006: 2252 -> 1604
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Some Other Race Alone Householder.SE_A14019_007: 605 -> 11
Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Two or More Races Householder.SE_A14019_008: 260 -> 0
American Indian and Alaska Native 16 Years Old in  Civilian Labor Fo

In [37]:
def inject_titles(col):
    """Function to inject titles into the column names of the ACS 1-year data."""
    key = col.split(".")[-1].split("_")[1]
    title = acs1_code_to_name["Title"].get(key, None)
    if title is not None:
        return title + "." + col
    return col


puma_df = puma_df.rename(columns=inject_titles)

In [38]:
# drop columns where measurements are suppressed or not available for any PUMA
old_columns = set(puma_df.columns)
puma_df = puma_df.dropna(axis=1)
for col in sorted(old_columns - set(puma_df.columns)):
    print(f"Dropped column: {col}")

Dropped column: Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) American Indian and Alaska Native Alone  Householder.SE_A14019_004
Dropped column: Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Asian Alone Householder.SE_A14019_005
Dropped column: Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Black or African American Alone Householder.SE_A14019_003
Dropped column: Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Hispanic or Latino Householder.SE_A14019_009
Dropped column: Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Native Hawaiian and Other Pacifi

In [39]:
# available columns
for col in sorted(puma_df.columns):
    print(col)

Age (Detailed Version) - Cumulative (Less).Total Less than 10 Years.SE_C01001A_003
Age (Detailed Version) - Cumulative (Less).Total Less than 15 Years.SE_C01001A_004
Age (Detailed Version) - Cumulative (Less).Total Less than 18 Years.SE_C01001A_005
Age (Detailed Version) - Cumulative (Less).Total Less than 20 Years.SE_C01001A_006
Age (Detailed Version) - Cumulative (Less).Total Less than 21 Years.SE_C01001A_007
Age (Detailed Version) - Cumulative (Less).Total Less than 22 Years.SE_C01001A_008
Age (Detailed Version) - Cumulative (Less).Total Less than 25 Years.SE_C01001A_009
Age (Detailed Version) - Cumulative (Less).Total Less than 30 Years.SE_C01001A_010
Age (Detailed Version) - Cumulative (Less).Total Less than 35 Years.SE_C01001A_011
Age (Detailed Version) - Cumulative (Less).Total Less than 40 Years.SE_C01001A_012
Age (Detailed Version) - Cumulative (Less).Total Less than 45 Years.SE_C01001A_013
Age (Detailed Version) - Cumulative (Less).Total Less than 5 Years.SE_C01001A_002
Age (

In [40]:
puma_migpuma = lio.load_puma_migpuma("../geometry/equivalencies/puma_migpuma_2010.csv")
puma_migpuma.head()

,State,MIGPUMA
PUMA,,
0100100,01,0100190
0100200,01,0100290
0100301,01,0100290
0100302,01,0100290
0100400,01,0100400


In [41]:
puma_df_numeric = puma_df.select_dtypes(include="number")
dropped = puma_df.columns.difference(puma_df.select_dtypes(include="number").columns)
print(dropped.tolist())

['File Identification.Geo_FILEID', 'Geographic Identifier.Geo_GEOID', 'Name of Area.Geo_NAME', 'Qualified Area Name.Geo_qname', 'Qualifying Name.Geo_QName', 'State/U.S.-Abbreviation (USPS).Geo_STUSAB']


In [42]:
puma_df_numeric["MIGPUMA"] = puma_migpuma.loc[puma_df_numeric.index, "MIGPUMA"]
puma_df_numeric["MIGPUMA"]

PUMA
0100100    0100190
0100200    0100290
0100301    0100290
0100302    0100290
0100400    0100400
            ...   
5600100    5600100
5600200    5600200
5600300    5600300
5600400    5600400
5600500    5600500
Name: MIGPUMA, Length: 2336, dtype: object

In [43]:
weight_col = "Total Population.Total Population.SE_A00001_001"

weighted_sum_cols = [
    c for c in puma_df_numeric.columns if "Median" in c or "Average" in c
]
other_cols = [
    c for c in puma_df_numeric.columns if c not in weighted_sum_cols and c != "MIGPUMA"
]


def agg_group(group):
    # usually can just sum
    result = {col: group[col].sum() for col in other_cols}
    # for aggregate statistics, take a weighted average
    result.update(
        {
            col: np.average(group[col], weights=group[weight_col])
            for col in weighted_sum_cols
        }
    )
    return pd.Series(result)


migpuma_df = puma_df_numeric.groupby("MIGPUMA").apply(agg_group, include_groups=False)
migpuma_df.head()

,Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).2.00 to 2.99.SE_C13004_003,Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).3.00 to 3.99.SE_C13004_004,Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).4.00 to 4.99.SE_C13004_005,Ratio of Income in 2018 to Poverty Level (Summarized - top-coded at 5.00).5.00 and Over.SE_C13004_006,Aggregate Family Income (In 2018 Inflation Adjusted Dollars).Aggregate Family Income (In 2018 Inflation Adjusted Dollars).SE_A14020_001,Aggregate Gross Rent.Aggregate Gross Rent for Specified Renter-Occupied Housing Units.SE_A18004_001,Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) Two or More Races Householder.SE_A14019_008,Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars) White Alone Householder.SE_A14019_002,Aggregate Household Income (In 2018 Inflation Adjusted Dollars).Aggregate Household Income (In 2018 Inflation Adjusted Dollars).SE_A14018_001,Aggregate Household Income (In 2018 Inflation Adjusted Dollars) by Race.Aggregate Household Income (In 2018 Inflation Adjusted Dollars).SE_A14019_001,...,Median Income by Sex by Work Experience (In 2018 Inflation Adjusted Dollars) (For Population Age 15+).Median Income for Population 15 Years and Over Female Other.SE_A14014_007,"Median Income by Sex by Work Experience (In 2018 Inflation Adjusted Dollars) (For Population Age 15+).Median Income for Population 15 Years and Over Female Worked Full-Time, Year-Round in the Past 12 Months.SE_A14014_006",Median Income by Sex by Work Experience (In 2018 Inflation Adjusted Dollars) (For Population Age 15+).Median Income for Population 15 Years and Over Female.SE_A14014_005,Median Income by Sex by Work Experience (In 2018 Inflation Adjusted Dollars) (For Population Age 15+).Median Income for Population 15 Years and Over Male Other.SE_A14014_004,"Median Income by Sex by Work Experience (In 2018 Inflation Adjusted Dollars) (For Population Age 15+).Median Income for Population 15 Years and Over Male Worked Full-Time, Year-Round in the Past 12 Months.SE_A14014_003",Median Income by Sex by Work Experience (In 2018 Inflation Adjusted Dollars) (For Population Age 15+).Median Income for Population 15 Years and Over Male.SE_A14014_002,Median Income by Sex by Work Experience (In 2018 Inflation Adjusted Dollars) (For Population Age 15+).Median Income for Population 15 Years and Over.SE_A14014_001,Median Nonfamily Household Income (In 2018 Inflation Adjusted Dollars).Median Nonfamily Household Income (In 2018 Inflation Adjusted Dollars).SE_A14012_001,Median House Value for All Owner-Occupied Housing Units.Median Value.SE_A10036_001,Median Year Structure Built.Median Year Structure Built.SE_A10057_001
MIGPUMA,,,,,,,,,,,,,,,,,,,,,
0100190,53617.0,43940.0,33416.0,50827.0,5.902934e+09,18469400.0,60220800.0,6.783205e+09,7.356363e+09,7.356363e+09,...,12081.241631,34213.459216,18723.363610,16799.893235,46316.796426,31194.552271,23992.719188,22355.024098,122260.006021,1978.776410
0100290,81728.0,74028.0,65170.0,161472.0,1.393221e+10,53273400.0,434355400.0,1.435536e+10,1.783981e+10,1.783981e+10,...,12849.410917,41425.061993,22063.943885,17881.037960,57435.400795,40741.624872,29661.532355,35612.030641,177615.203775,1988.892087
0100400,20418.0,15526.0,11217.0,17273.0,2.247872e+09,5911200.0,59444000.0,2.429900e+09,2.642520e+09,2.642520e+09,...,11552.000000,29585.000000,16501.000000,16881.000000,42294.000000,29656.000000,22030.000000,19562.000000,107500.000000,1983.000000
0100600,26893.0,21038.0,15722.0,33435.0,3.292696e+09,11148400.0,18863500.0,3.325720e+09,3.918489e+09,3.918489e+09,...,12451.000000,35014.000000,20796.000000,17911.000000,45994.000000,35200.000000,26886.000000,28353.000000,132500.000000,1983.000000
0100700,23592.0,10818.0,8018.0,19375.0,1.990020e+09,5155600.0,24876700.0,2.430421e

In [44]:
def median_from_binned(counts, edges):
    """
    Compute median from binned frequency data via linear interpolation.

    counts : array-like of frequencies for each bin, in order
    edges  : array-like of bin edges, length = len(counts) + 1
             e.g. bins [0-10, 10-20, 20-30] -> edges = [0, 10, 20, 30]
    """
    counts = np.array(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return np.nan

    half = total / 2
    cum = np.cumsum(counts)

    # index of the bin where the cumulative count first reaches/exceeds half
    median_bin_idx = np.searchsorted(cum, half)
    cf_before = cum[median_bin_idx - 1] if median_bin_idx > 0 else 0
    f = counts[median_bin_idx]
    L = edges[median_bin_idx]
    h = edges[median_bin_idx + 1] - L

    if f == 0:
        return L  # avoid divide-by-zero if the median bin is empty

    return L + ((half - cf_before) / f) * h


bin_cols_travel_time = [
    "Travel Time to Work for Workers 16 Years and Over (Who Did Not Work At Home).Workers 16 Years and Over Who Did Not Work At Home Less than 10 Minutes.SE_B09001_002",
    "Travel Time to Work for Workers 16 Years and Over (Who Did Not Work At Home).Workers 16 Years and Over Who Did Not Work At Home 10 to 19 Minutes.SE_B09001_003",
    "Travel Time to Work for Workers 16 Years and Over (Who Did Not Work At Home).Workers 16 Years and Over Who Did Not Work At Home 20 to 29 Minutes.SE_B09001_004",
    "Travel Time to Work for Workers 16 Years and Over (Who Did Not Work At Home).Workers 16 Years and Over Who Did Not Work At Home 30 to 39 Minutes.SE_B09001_005",
    "Travel Time to Work for Workers 16 Years and Over (Who Did Not Work At Home).Workers 16 Years and Over Who Did Not Work At Home 40 to 59 Minutes.SE_B09001_006",
    "Travel Time to Work for Workers 16 Years and Over (Who Did Not Work At Home).Workers 16 Years and Over Who Did Not Work At Home 60 to 89 Minutes.SE_B09001_007",
    "Travel Time to Work for Workers 16 Years and Over (Who Did Not Work At Home).Workers 16 Years and Over Who Did Not Work At Home 90 or More Minutes.SE_B09001_008",
]

# Bin edges — last bin is an estimate
edges_travel_time = [0, 10, 20, 30, 40, 60, 90, 120]

bin_cols_rent_pct = [
    "Gross Rent as a Percentage of Household Income in 2018.Renter-Occupied Housing Units Less than 10 Percent.SE_A18002_002",
    "Gross Rent as a Percentage of Household Income in 2018.Renter-Occupied Housing Units 10 to 29 Percent.SE_A18002_003",
    "Gross Rent as a Percentage of Household Income in 2018.Renter-Occupied Housing Units 30 to 49 Percent.SE_A18002_004",
    "Gross Rent as a Percentage of Household Income in 2018.Renter-Occupied Housing Units 50 Percent or More.SE_A18002_005",
]
# last one is an estimate
edges_rent_pct = [10, 30, 50, 75]

bin_cols_owner_cost_pct = [
    "Selected Monthly Owner Costs as a Percentage of Household Income in 2018 for Housing Units Without a Mortgage.Housing Units Without a Mortgage Less than 30 Percent.SE_A10051_002",
    "Selected Monthly Owner Costs as a Percentage of Household Income in 2018 for Housing Units Without a Mortgage.Housing Units Without a Mortgage 30 Percent or More.SE_A10051_003",
    "Selected Monthly Owner Costs as a Percentage of Household Income in 2018 for Housing Units Without a Mortgage.Housing Units Without a Mortgage 50 Percent or More.SE_A10051_004",
]
edges_owner_cost_pct = [30, 50, 75]


In [45]:
for x in sorted(migpuma_df.columns):
    print(x)

Age (Detailed Version) - Cumulative (Less).Total Less than 10 Years.SE_C01001A_003
Age (Detailed Version) - Cumulative (Less).Total Less than 15 Years.SE_C01001A_004
Age (Detailed Version) - Cumulative (Less).Total Less than 18 Years.SE_C01001A_005
Age (Detailed Version) - Cumulative (Less).Total Less than 20 Years.SE_C01001A_006
Age (Detailed Version) - Cumulative (Less).Total Less than 21 Years.SE_C01001A_007
Age (Detailed Version) - Cumulative (Less).Total Less than 22 Years.SE_C01001A_008
Age (Detailed Version) - Cumulative (Less).Total Less than 25 Years.SE_C01001A_009
Age (Detailed Version) - Cumulative (Less).Total Less than 30 Years.SE_C01001A_010
Age (Detailed Version) - Cumulative (Less).Total Less than 35 Years.SE_C01001A_011
Age (Detailed Version) - Cumulative (Less).Total Less than 40 Years.SE_C01001A_012
Age (Detailed Version) - Cumulative (Less).Total Less than 45 Years.SE_C01001A_013
Age (Detailed Version) - Cumulative (Less).Total Less than 5 Years.SE_C01001A_002
Age (

In [ ]:
for table in [migpuma_df, puma_df]:
    table["Unemployment rate"] = (
        table[
            "Unemployment Rate for Civilian Population in Labor Force 16 Years and Over.Civilian Population in Labor Force 16 Years and Over Unemployed.SE_A17005_003"
        ]
        / table[
            "Unemployment Rate for Civilian Population in Labor Force 16 Years and Over.Civilian Population in Labor Force 16 Years and Over.SE_A17005_001"
        ]
    )
    table[
        "Unemployment Rate for Civilian Population in Labor Force 16 Years and Over.Civilian Population in Labor Force 16 Years and Over Unemployed.SE_A17005_003"
    ]
    table["Proportion of people in college"] = (
        table[
            "Level of School by Type of School for Enrolled in School Population 3 Years and Over.Population 3 Years and Over Enrolled in School Private School College.SE_A12006_011"
        ]
        + table[
            "Level of School by Type of School for Enrolled in School Population 3 Years and Over.Population 3 Years and Over Enrolled in School Public School College.SE_A12006_006"
        ]
    ) / table["Total Population.Total Population.SE_A00001_001"]
    table["House vacancy proportion"] = (
        table["Occupancy Status.Housing Units Vacant.SE_A10044_003"]
        / table["Occupancy Status.Housing Units.SE_A10044_001"]
    )

    table["Median travel time"] = table[bin_cols_travel_time].apply(
        lambda row: median_from_binned(row.values, edges_travel_time), axis=1
    )
    table["Median gross rent as a percentage of household income"] = table[
        bin_cols_rent_pct
    ].apply(lambda row: median_from_binned(row.values, edges_rent_pct), axis=1)

    table["Median selected monthly owner costs as percentage of household income"] = (
        table[bin_cols_owner_cost_pct].apply(
            lambda row: median_from_binned(row.values, edges_owner_cost_pct), axis=1
        )
    )
    table["Median house value over median household income"] = (
        table[
            "Median House Value for All Owner-Occupied Housing Units.Median Value.SE_A10036_001"
        ]
        / table[
            "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001"
        ]
    )

    table["Proportion of people White"] = (
        table["Total Population Not Hispanic or Latino White Alone.SE_B04001_003"]
        / table["Total Population.SE_B04001_001"]
    )
    table["Proportion of people Asian"] = (
        table["Total Population Not Hispanic or Latino Asian Alone.SE_B04001_006"]
        / table["Total Population.SE_B04001_001"]
    )
    table["Proportion of people PI"] = (
        table[
            "Total Population Not Hispanic or Latino Native Hawaiian and Other Pacific Islander Alone.SE_B04001_007"
        ]
        / table["Total Population.SE_B04001_001"]
    )
    table["Proportion of people AAPI"] = (
        table["Total Population Not Hispanic or Latino Asian Alone.SE_B04001_006"]
        + table[
            "Total Population Not Hispanic or Latino Native Hawaiian and Other Pacific Islander Alone.SE_B04001_007"
        ]
    ) / table["Total Population.SE_B04001_001"]
    table["Proportion of people Black"] = (
        table[
            "Total Population Not Hispanic or Latino Black or African American Alone.SE_B04001_004"
        ]
        / table["Total Population.SE_B04001_001"]
    )
    table["Proportion of people Latino"] = (
        table["Total Population Hispanic or Latino.SE_B04001_010"]
        / table["Total Population.SE_B04001_001"]
    )
    table["Proportion of people Indian"] = (
        table[
            "Total Population Not Hispanic or Latino American Indian and Alaska Native Alone.SE_B04001_005"
        ]
        / table["Total Population.SE_B04001_001"]
    )

    table["Proportion of people other race"] = (
        table[
            "Total Population Not Hispanic or Latino Some Other Race Alone.SE_B04001_008"
        ]
        + table[
            "Total Population Not Hispanic or Latino Two or More Races.SE_B04001_009"
        ]
    ) / table["Total Population.SE_B04001_001"]

    table["Proportion of people struggling"] = (
        table[
            "Ratio of Income in 2018 to Poverty Level (Summarized).Population for Whom Poverty Status Is Determined Under 2.00 (Poor or Struggling).SE_B13004_004"
        ]
        / table[
            "Ratio of Income in 2018 to Poverty Level (Summarized).Population for Whom Poverty Status Is Determined.SE_B13004_001"
        ]
    )
    table["Years since median structure built"] = (
        year
        - table["Median Year Structure Built.Median Year Structure Built.SE_A10057_001"]
    )
    table["Proportion foreign born"] = (
        table[
            "Nativity by Citizenship Status.Total Population Foreign Born.SE_A06001_003"
        ]
        / table["Nativity by Citizenship Status.Total Population.SE_A06001_001"]
    )

    table["Proportion alternative commute"] = (
        table[
            "Means of Transportation to Work (Renter-Occupied Housing Units).Workers 16 Years and Over in Households Householder Lived in Renter-Occupied Housing Units Public Transportation (Excluding Taxicab).SE_A09004B_004"
        ]
        + table[
            "Means of Transportation to Work (Renter-Occupied Housing Units).Workers 16 Years and Over in Households Householder Lived in Renter-Occupied Housing Units Walked.SE_A09004B_005"
        ]
    ) / table[
        "Means of Transportation to Work (Renter-Occupied Housing Units).Workers 16 Years and Over in Households.SE_A09004B_008"
    ]
    table["Proportion WFH"] = (
        table[
            "Means of Transportation to Work (Renter-Occupied Housing Units).Workers 16 Years and Over in Households Householder Lived in Renter-Occupied Housing Units Worked At Home.SE_A09004B_007"
        ]
        / table[
            "Means of Transportation to Work (Renter-Occupied Housing Units).Workers 16 Years and Over in Households.SE_A09004B_008"
        ]
    )

    table["Proportion of households with children"] = (
        table[
            "Housing Units by Families and Presence of Own Children.Occupied Housing Units with Related Children of the Householder Under 18.SE_A10022_002"
        ]
        / table[
            "Housing Units by Families and Presence of Own Children.Occupied Housing Units.SE_A10022_001"
        ]
    )
    table["Labor force participation rate"] = (
        table[
            "Employment Status for Total Population 16 Years and Over.Population 16 Years and Over in Labor Force.SE_A17002_002"
        ]
        / (
            table[
                "Employment Status for Total Population 16 Years and Over.Population 16 Years and Over.SE_A17002_001"
            ]
        )
    )
    table["Proportion of people in military"] = (
        table[
            "Labor Force for Population 16 Years and Over.In Labor Force 16 Years and Over in Armed Forces.SE_A17001_002"
        ]
        / table[
            "Labor Force for Population 16 Years and Over.In Labor Force 16 Years and Over.SE_A17001_001"
        ]
    )

    table["Proportion of people 18-34"] = (
        table["Age (Short Version).Total Population 18 to 34 Years.SE_B01001_003"]
        / table["Age (Short Version).Total Population.SE_B01001_001"]
    )
    table["Proportion of people 35-64"] = (
        table["Age (Short Version).Total Population 35 to 64 Years.SE_B01001_004"]
        / table["Age (Short Version).Total Population.SE_B01001_001"]
    )
    table["Proportion of people 65+"] = (
        table["Age (Short Version).Total Population 65 and Over.SE_B01001_005"]
        / table["Age (Short Version).Total Population.SE_B01001_001"]
    )
    table["Proportion of people under 18"] = (
        table["Age (Short Version).Total Population Under 18 Years.SE_B01001_002"]
        / table["Age (Short Version).Total Population.SE_B01001_001"]
    )

    table["Proportion of people receiving public assistance income"] = (
        table[
            "Households with Public Assistance Income.Households with Public Assistance Income.SE_A10014_002"
        ]
        / table["Households with Public Assistance Income.Households.SE_A10014_001"]
    )
    table["Median household income in tens of thousands of dollars"] = (
        table[
            "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001"
        ]
        / 10_000
    )
    table["Median house cost in hundreds of thousands of dollars"] = (
        table[
            "Median House Value for All Owner-Occupied Housing Units.Median Value.SE_A10036_001"
        ]
        / 100_000
    )
    table["Median gross rent in thousands of dollars"] = (
        table["Median Gross Rent.Median Gross Rent.SE_A18009_001"] / 1_000
    )
    table["Median earnings in tens of thousands of dollars"] = (
        table[
            "Median Earnings by Sex by Educational Attainment in the past 12 months (In 2018 Inflation Adjusted Dollars) (For Population Age 25+).Median Earnings.SE_A15001_001"
        ]
        / 10_000
    )
    table["TOT_POP"] = table["Total Population.Total Population.SE_A00001_001"]

In [47]:
migpuma_df.to_csv(f"acs_migpuma_{year}.csv")
puma_df.to_csv(f"acs_puma_{year}.csv")